#### Preparing Data

In [ ]:
# import the necessary packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

import keras

In [ ]:
# reading csv data
columns = [
    "capacity",
    "diameter",
    "column_height",
    "porosity",
    "ads_time",
    "C_o",
    "total_LCP",
    "emissions",
    "PFAS_eff",
    "PFAS_ads",
    ]

df1 = pd.read_csv("./training_data/GAC_60k_CORRECTED.csv", usecols=columns)
df = df1.dropna(ignore_index=True)
# df.tail()

In [ ]:
# count rows before cleaning
print(f"Number of rows before cleaning: {len(df)}")

In [ ]:
epsilon = 1e-20
# cleaning data
df.loc[:, 'PFAS_eff'] = df['PFAS_eff'].clip(lower=0)
df = df[(df['total_LCP'] >= 0)].copy()
df['total_LCP'] = np.log(df['total_LCP'])
df['PFAS_eff'] = np.log(df['PFAS_eff'] + epsilon)

# bounds
print(df.min())
print(df.max())

In [ ]:
# count rows after cleaning
print(f"Number of rows after cleaning: {len(df)}")

In [ ]:
# separate the data into inputs and outputs
inputs = ["capacity", "diameter", "column_height", "porosity", "ads_time", "C_o"]
outputs = [
    "total_LCP",
    "emissions",
    "PFAS_eff",
]
dfin = df[inputs]
dfout = df[outputs]

In [ ]:
# define a function to scale the data using min-max scaling
def min_max_scaler(data, data_min, data_max):
    return (data - data_min) /(data_max - data_min)

# define a function to reverse the scaling
def reverse_min_max_scaler(scaled_data, data_min, data_max):
    return scaled_data * (data_max - data_min) + data_min

In [ ]:
# scale the data
x_data_min = dfin.min()
x_data_max = dfin.max()
y_data_min = dfout.min()
y_data_max = dfout.max()

x_data_scaled = min_max_scaler(dfin, x_data_min, x_data_max)
y_data_scaled = min_max_scaler(dfout, y_data_min, y_data_max)

x_scaled_min = x_data_scaled.min()
x_scaled_max = x_data_scaled.max()
y_scaled_min = y_data_scaled.min()
y_scaled_max = y_data_scaled.max()


#### Optimise Model with pyomo

In [ ]:
import pyomo.environ as pyo
# omlt for interfacing our neural network with pyomo
from Custom_scaling import MinMaxScaling
from omlt import OffsetScaling, OmltBlock
from omlt.io.keras import keras_reader, load_keras_sequential
from omlt.neuralnet import (
    FullSpaceNNFormulation,
    FullSpaceSmoothNNFormulation,
    ReducedSpaceSmoothNNFormulation,
    ReluBigMFormulation,
    ReluComplementarityFormulation,
    ReluPartitionFormulation,
)
from omlt.neuralnet.activations import ComplementarityReLUActivation

In [ ]:
# # Check solver availability
# for solver in ["glpk", "cbc", "ipopt", "baron", "cplex", "appsi_maingo", "gurobi"]:
#     print(solver, pyo.SolverFactory(solver).available())

In [ ]:
# unscaling back using factors
scaler = MinMaxScaling(
        min_inputs={i: x_data_min[inputs[i]] for i in range(len(inputs))},
        max_inputs={i: x_data_max[inputs[i]] for i in range(len(inputs))},
        min_outputs={i: y_data_min[outputs[i]] for i in range(len(outputs))},
        max_outputs={i: y_data_max[outputs[i]] for i in range(len(outputs))},
    )

# saving unscaled bounds
#input
unscaled_lb = x_data_min[inputs].to_numpy()
unscaled_ub = x_data_max[inputs].to_numpy()
#output
output_lb = y_data_min[outputs].to_numpy()
output_ub = y_data_max[outputs].to_numpy()

# saving scaled bounds
# input
scaled_lb = x_scaled_min[inputs].to_numpy()
scaled_ub = x_scaled_max[inputs].to_numpy()
# output
scaled_output_lb = y_scaled_min[outputs].to_numpy()
scaled_output_ub = y_scaled_max[outputs].to_numpy()

scaled_input_bounds = {i: (scaled_lb[i], scaled_ub[i]) for i in range(len(inputs))}
unscaled_input_bounds = {i: (unscaled_lb[i], unscaled_ub[i]) for i in range(len(inputs))}

In [ ]:
# Optimisation Data
obj_scaler = 1e-6
feed_conc_val = 0.5 # ppb
EPA_conc_val = 9.66e-9 # mol/m3
capacity_val = 115 # m3/h
porosity_val = 0.4

# loading saved nn
nn_PFAS_model = keras.models.load_model("GAC_sig_mse(30,30,30)_100k.keras", compile=False)
net = load_keras_sequential(
        nn_PFAS_model, scaling_object=scaler, scaled_input_bounds=scaled_input_bounds, unscaled_input_bounds=unscaled_input_bounds
    )

# Pyomo model with algebraic constraints
def solve_model(objective='cost', eps_emissions=None, feed_conc=feed_conc_val, EPA_conc=EPA_conc_val, capacity=capacity_val, porosity=porosity_val):

    optimization_model = pyo.ConcreteModel()
    
    # Define free variables (Inputs the solver must optimize)
    optimization_model.diameter = pyo.Var(initialize=(unscaled_lb[1]+unscaled_ub[1])/2, bounds=(unscaled_lb[1], unscaled_ub[1]))
    optimization_model.column_height = pyo.Var(initialize=(unscaled_lb[2]+unscaled_ub[2])/2, bounds=(unscaled_lb[2], unscaled_ub[2]))
    optimization_model.porosity = pyo.Var(initialize=(unscaled_lb[3]+unscaled_ub[3])/2, bounds=(unscaled_lb[3], unscaled_ub[3]))
    optimization_model.ads_time = pyo.Var(initialize=(unscaled_lb[4]+unscaled_ub[4])/2, bounds=(unscaled_lb[4], unscaled_ub[4]))
    
    # Output Variables
    optimization_model.total_LCP = pyo.Var(bounds=(output_lb[0], output_ub[0]))
    optimization_model.emissions = pyo.Var(bounds=(output_lb[1], output_ub[1]))
    optimization_model.PFAS_eff_conc = pyo.Var(bounds=(output_lb[2], output_ub[2]))

    # OMLT Block
    optimization_model.nn = OmltBlock()
    formulated_problem = FullSpaceSmoothNNFormulation(net)
    optimization_model.nn.build_formulation(formulated_problem)
    
    # Fix the known inputs directly to shrink the problem size
    optimization_model.nn.inputs[0].fix(capacity)
    optimization_model.nn.inputs[5].fix(feed_conc)

    # Link only the free variables
    optimization_model.con_in_d = pyo.Constraint(expr= optimization_model.diameter == optimization_model.nn.inputs[1])
    optimization_model.con_in_H = pyo.Constraint(expr= optimization_model.column_height == optimization_model.nn.inputs[2])
    optimization_model.con_in_p = pyo.Constraint(expr= optimization_model.porosity == optimization_model.nn.inputs[3])
    optimization_model.con_in_t = pyo.Constraint(expr= optimization_model.ads_time == optimization_model.nn.inputs[4])

    # Link the outputs
    optimization_model.con_out_LCP = pyo.Constraint(expr= optimization_model.total_LCP == optimization_model.nn.outputs[0])
    optimization_model.con_out_emi = pyo.Constraint(expr= optimization_model.emissions == optimization_model.nn.outputs[1])
    optimization_model.con_out_eff = pyo.Constraint(expr= optimization_model.PFAS_eff_conc == optimization_model.nn.outputs[2])

    # Performance Constraints
    optimization_model.epa_limit = pyo.Constraint(expr= optimization_model.PFAS_eff_conc <= pyo.log(EPA_conc + epsilon))
    
    if eps_emissions is not None:
        optimization_model.emissions_con = pyo.Constraint(expr= optimization_model.emissions <= eps_emissions)
    
    # Objective
    if objective == 'cost':
        optimization_model.obj = pyo.Objective(expr= optimization_model.total_LCP, sense=pyo.minimize)
    else:
        optimization_model.obj = pyo.Objective(expr= optimization_model.emissions, sense=pyo.minimize)

    # BARON Configurations
    options = {
    'MaxTime': 3600,   # Maximum time in seconds
    'epsr': 0.05,      # relative gap (percentage fraction)
    'epsa': 1e-4,      # Absolute precision
    # 'threads': 4,     # No of worker threads for parallel processing     
    'prlevel': 1,      # 1 = Print normal logs, 0 = silent      
    'dolocal': 1       # 1 = Use local search heuristics to speed up finding bounds     
    }
    
    solver = pyo.SolverFactory('baron') 
    results = solver.solve(optimization_model, tee=True, options=options, keepfiles=True)

    print("# of variables: ", optimization_model.nvariables())
    print("# of constraints: ", optimization_model.nconstraints())

    term_cond = str(results.solver.termination_condition)
    
    # Silently check if the objective has a value
    obj_val = pyo.value(optimization_model.obj, exception=False)
    solution_found = (obj_val is not None)

    if term_cond == 'optimal':
        print("Success: Global optimal solution found")
        status = "Optimal"
    elif term_cond == 'maxTimeLimit' and solution_found:
        print(f"Success: Time limit hit. Feasible solution found (Objective: {obj_val:.4f}).")
        status = "Feasible (Time Limit)"
    else:
        print(f"Solver stopped ('{term_cond}') and NO feasible solution was found.")
        status = "Infeasible"

    def get_val(var):
        # Silently check each variable, return NaN if empty
        v = pyo.value(var, exception=False)
        return v if v is not None else np.nan

    return (capacity,
            get_val(optimization_model.diameter),
            get_val(optimization_model.column_height),
            get_val(optimization_model.porosity),
            get_val(optimization_model.ads_time),
            feed_conc,
            np.exp(get_val(optimization_model.total_LCP)),
            get_val(optimization_model.emissions),
            np.exp(get_val(optimization_model.PFAS_eff_conc) - epsilon),
            status)

In [ ]:
MW_PFAS = 414.07  
def get_effluent_mol_m3(concentration_ppb):
    return (concentration_ppb * 1e-3) / MW_PFAS

epa_limits_ppb = [0.07, 0.02, 0.004, 0.001, 0.0001] 

# Assuming unscaled_lb and unscaled_ub are defined earlier
capacities = np.linspace(unscaled_lb[0], unscaled_ub[0], 10)

heatmap1_data = []
full_solutions_data = []

print("Running Experiment 1: Policy Impact...")

for q in capacities:
    for epa_ppb in epa_limits_ppb:
        print(f"Solving Q={q:.1f}, EPA={epa_ppb} ppb")
        epa_mol_m3 = get_effluent_mol_m3(epa_ppb)
        
        # Unpack the 10 results from solve_model
        res = solve_model(objective='cost', capacity=q, EPA_conc=epa_mol_m3)
        capacity, diameter, height, porosity, ads_time, C_o, cost, emissions, eff_conc, status = res
        
        # 1. Original data for plotting (Heatmap format)
        heatmap1_data.append({
            'Capacity_m3h': q, 
            'EPA_Limit_ppb': epa_ppb, 
            'Cost': cost
        })
        
        # 2. Full solution data for the engineering records
        full_solutions_data.append({
            'Capacity_m3h': capacity,
            'EPA_Limit_ppb': epa_ppb,
            'Optimal_Diameter_m': diameter,
            'Optimal_Height_m': height,
            'Bed_Porosity': porosity,
            'Ads_Time': ads_time,
            'Feed_Conc_ppb': C_o,
            'Levelized_Cost_$/kg': cost,
            'Emissions_kgCO2/hr': emissions,
            'PFAS_eff': eff_conc,
            'Status': status
        })

# Create DataFrames
df_plot = pd.DataFrame(heatmap1_data)
# Pivot the plotting data into the 2D matrix format for the heatmap
df_heatmap_matrix = df_plot.pivot(index='Capacity_m3h', columns='EPA_Limit_ppb', values='Cost')

# Create the full solutions DataFrame
df_full_solutions = pd.DataFrame(full_solutions_data)

# Save BOTH DataFrames to a single Excel file using multiple sheets
excel_filename = 'Cost_vs_Capacity_and_EffluentLimit.xlsx'
with pd.ExcelWriter(excel_filename, engine='openpyxl') as writer:
    # Sheet 1: The pivoted matrix for easy plotting in OriginPro/Python
    df_heatmap_matrix.to_excel(writer, sheet_name='Plotting_Matrix')
    
    # Sheet 2: The raw, unpivoted full solutions for every variable
    df_full_solutions.to_excel(writer, sheet_name='Full_Solutions', index=False)

print(f"Saved Experiment 1 to {excel_filename} (Contains 'Plotting_Matrix' and 'Full_Solutions' sheets)")